In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy
from dotenv import load_dotenv
import os
from tqdm import tqdm
import tensorstore as ts

load_dotenv()
PATH = os.getenv("ROOT_PATH")

plt.style.use(['science', 'no-latex'])

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
subject_id = "06"

traces = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_traces.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

coordinates = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_coordinates.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

traces = traces.read().result()
coordinates = coordinates.read().result()

In [ ]:
import h5py
import numpy as np
f = h5py.File(f'/{PATH}/Additional_mat_files/MaskDatabase.mat', 'r')
names = f['MaskDatabaseNames']
names = [(i, "".join([chr(c[0]) for c in f[name[0]]])) for i, name in enumerate(names)]

In [ ]:
def linear_to_3d_matlab(linear_idx, width, height):
  idx = linear_idx - 1
  i = idx % width
  j = (idx // width) % height
  k = idx // (width * height)
  return i, j, k

def matlab_to_linear(i, j, k, width, height):
  linear_idx = i + j * width + k * width * height + 1
  return linear_idx


reference_anat = scipy.io.loadmat(f'/{PATH}/Additional_mat_files/ReferenceBrain.mat')['anat_stack_norm']

ir = f['MaskDatabase']['ir'][:]  # Row indices (linear voxel indices)
jc = f['MaskDatabase']['jc'][:]  # Column pointers
data = f['MaskDatabase']['data'][:]  # Should be all ones

mask = np.zeros_like(reference_anat)
mask_shape = mask.shape

# mask_idx_list = [77, 66, 67, 68, 69, 70, 71, 101, 102, 103, 106]
mask_idx_list = [106] # visual processing
# mask_idx_list = [96, 97, 200] # eye movement control
for mask_idx in mask_idx_list:
  start_idx = jc[mask_idx]
  end_idx = jc[mask_idx + 1]
  for i in ir[start_idx:end_idx]:
    i_x, i_y, i_z = linear_to_3d_matlab(i, mask_shape[0], mask_shape[1])
    mask[i_x, i_y, i_z] = 1

In [ ]:
import pyvista as pv
import matplotlib.pyplot as plt
import numpy as np

reference_anat[mask==0]=0

grid = pv.wrap(reference_anat)
grid.spacing = [1.0, 1.0, 2.0]
valid_coordinates = []
valid_coordinate_i = []
for i, c in enumerate(coordinates):
  if mask[c.astype(int)[0], c.astype(int)[1], c.astype(int)[2]]:
    valid_coordinates.append(c)
    valid_coordinate_i.append(i)
valid_coordinates = np.stack(valid_coordinates)
valid_coordinate_i = np.array(valid_coordinate_i)[np.mean(traces[..., valid_coordinate_i], 0)>-1]
points = valid_coordinates * np.array(grid.spacing)
colors = np.random.rand(len(points))
point_cloud = pv.PolyData(points)

In [ ]:
pv.set_jupyter_backend('static')
plotter = pv.Plotter(notebook=True)
plotter.add_volume(
    grid,
    cmap="gray",
    opacity=[0.0,0.1,0.3,0.5],
    shade=True,
    show_scalar_bar=False,
)

colors=np.mean(traces[..., valid_coordinate_i], 0)
plotter.add_mesh(
    point_cloud,
    scalars=colors,
    cmap='viridis',
    point_size=5,
    render_points_as_spheres=True,
    show_scalar_bar=False
)

plotter.camera_position = 'xy'
plotter.camera.azimuth = 0
plotter.camera.elevation = 0
plotter.camera.zoom(1.5)
plotter.show()

In [ ]:
coordinates_left_hemisphere = valid_coordinate_i[(coordinates[valid_coordinate_i]<300)[:, 1]]
coordinates_right_hemisphere = valid_coordinate_i[(coordinates[valid_coordinate_i]>300)[:, 1]]

In [ ]:
coordinates_left_hemisphere

In [ ]:
n_t, n_neurons = traces[..., coordinates_left_hemisphere].shape

fig, axs = plt.subplots(10, figsize=(10, 10), dpi=200)
for i in range(0, 10):
  ax = axs[i]
  ax.plot(traces[..., coordinates_left_hemisphere][..., i],'k',linewidth=1)
  format_ax(ax)
  # ax.set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
valid_coordinate_i

knn predictability

In [ ]:
context = 4
horizon = 50
n_neurons = traces.shape[-1]
traces.shape[0]-context-horizon
trace_data_x = np.stack([traces[i:i+context] for i in range(0, traces.shape[0]-context-horizon, 100)])
trace_data_theta = np.stack([traces[i+context+horizon] for i in range(0, traces.shape[0]-context-horizon, 100)])
trace_data_x_regional = np.stack([traces[..., valid_coordinate_i][i:i+context] for i in range(0, traces[..., valid_coordinate_i].shape[0]-context-horizon, 100)])
trace_data_theta_regional = np.stack([traces[..., valid_coordinate_i][i+context+horizon] for i in range(0, traces[..., valid_coordinate_i].shape[0]-context-horizon, 100)])

In [ ]:
trace_data_x = trace_data_x.transpose(0, 2, 1).reshape(trace_data_x.shape[0]*n_neurons, context)
trace_data_theta = trace_data_theta.ravel()
trace_data_x_regional = trace_data_x_regional.transpose(0, 2, 1).reshape(trace_data_x_regional.shape[0]*trace_data_x_regional.shape[2], context)
trace_data_theta_regional = trace_data_theta_regional.ravel()

In [ ]:
i = 0
plt.figure(dpi=200)
plt.scatter(trace_data_x[::10, 0], trace_data_x[::10, 1],c="k",marker=".",s=1)
plt.scatter(trace_data_x_regional[::, 0], trace_data_x_regional[::, 1],c="r",marker=".",s=1)

In [ ]:
def find_closest_theta(x, traces_x, traces_theta, n = 50):
  distances = np.linalg.norm(traces_x - x, axis=1)
  closest_ix = np.argsort(distances)[:n]
  closest_theta = traces_theta[closest_ix]
  return closest_theta

for k in range(0, 10):
  predicted_trace_i = []
  theta_hat_i = []
  for j in tqdm(range(0, 500)):
    theta_hat = find_closest_theta(traces[:, valid_coordinate_i[k]][j:j+context], trace_data_x_regional, trace_data_theta_regional)
    # theta_hat = find_closest_theta(traces[:, valid_coordinate_i[k]][j:j+context], trace_data_x[::200], trace_data_theta[::200])
    theta_hat_i.append(theta_hat)
    predicted_trace_i.append(theta_hat.mean())
  theta_hat_i = np.array(theta_hat_i)
  plt.figure(figsize=(10, 1), dpi=500)
  plt.plot(traces[context:500, valid_coordinate_i[k]], 'k')
  plt.plot(np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
  plt.fill_between(
    np.arange(theta_hat_i.shape[0]),
    np.quantile(theta_hat_i, 0.05, axis=-1),
    np.quantile(theta_hat_i, 0.95, axis=-1),
    color='red',
    alpha=0.1,
    linewidth=0,
  );
  # plt.xlim(0, 300)
  plt.ylim(0, 1)